# BRISC 2025 - Comparative analysis and figure generation
**Catarina Bota | UCP**

This notebook reads all outputs generated by the 3 workflows and produces the final figures for the article.

**Workflows (new order):**
1. **Workflow 1: Traditional ML** — Wavelet + GLCM + LightGBM (91.38%)
2. **Workflow 2: Deep Learning** — End-to-end CNN (94.54%)
3. **Workflow 3: Hybrid** — CNN features + XGBoost (96.50%)

**Outputs:** PNG + PDF (vector) in the `latex_artigo/figures/` folder.

## 1. Imports and shared style

Academic teal/sage palette + serif font (consistent with the UCP template).

In [ ]:
import os, glob, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# --- Style ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.titlecolor': '#222',
    'axes.titlelocation': 'left',
    'axes.titlepad': 12,
    'legend.fontsize': 9.5,
    'legend.frameon': False,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.edgecolor': '#888',
    'figure.dpi': 150,
})

# --- Palette ---
PALETTE = {
    'ML':       '#8FA89E',   # light sage (Workflow 1 - classical)
    'CNN':      '#4A8584',   # medium teal (Workflow 2 - DL)
    'Hybrid':   '#1F4E4A',   # deepest forest (Workflow 3 - best)
    'good':     '#2E6B68',
    'bad':      '#B5CCC4',
    'neutral1': '#4A8584',
    'neutral2': '#BFD1CA',
    'accent':   '#1F4E4A',
    'gap':      '#9AA8A0',
}

def style_axis(ax, keep_y=False):
    for s in ['top','right','left']:
        ax.spines[s].set_visible(False)
    if keep_y:
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color('#888')
        ax.tick_params(axis='y', length=3)
    else:
        ax.tick_params(axis='y', length=0)
    ax.spines['bottom'].set_color('#888')
    ax.set_axisbelow(True)

print('Style and palette ready.')

## 2. Paths and directories

Project folders are detected automatically.

In [ ]:
BASE   = r'C:\Users\cbot\Desktop\BRISC pos graduação'
WF3    = os.path.join(BASE, 'workflow3_resultados')
OUT    = os.path.join(BASE, 'latex_artigo', 'figures')
MATLAB = r'C:\Users\cbot\Documents\MATLAB\classification_task\all\resultados'
CNN_XL = os.path.join(BASE, 'Model_Metrics_Report.xlsx')

os.makedirs(OUT, exist_ok=True)
for p, label in [(BASE,'BASE'), (WF3,'WF3'), (OUT,'OUT'), (MATLAB,'MATLAB'), (CNN_XL,'CNN_XL')]:
    print(f'  [{label:6s}] {p}  ->  {"OK" if os.path.exists(p) else "MISSING"}')

## 3. Load the data from the 3 workflows

- **Workflow 1 (ML)**: 16 Excel files with the factorial grid.
- **Workflow 2 (CNN)**: `Model_Metrics_Report.xlsx`.
- **Workflow 3 (Hybrid)**: CSVs in `workflow3_resultados/`.

In [ ]:
# --- Workflow 1: ML (Wavelet+GLCM+LGBM) ---
ml_files = sorted(glob.glob(os.path.join(MATLAB, 'Relatorio_*.xlsx')))
ml_rows = []
for f in ml_files:
    name = os.path.basename(f).replace('Relatorio_', '').split('_2026')[0]
    if 'local' in name:
        continue
    df = pd.read_excel(f, sheet_name='Main_Results')
    df['scenario'] = name
    ml_rows.append(df)
ml = pd.concat(ml_rows, ignore_index=True)
ml_best = ml.sort_values('f1_score', ascending=False).groupby('scenario').head(1).sort_values('f1_score', ascending=False)
print(f'Workflow 1: {len(ml_files)} Excel files, {len(ml)} runs total, best per scenario:')
print(ml_best[['scenario','classificador','NumLevels','split','accuracy','f1_score','auc']].to_string(index=False))

In [ ]:
# --- Workflow 2: CNN ---
cnn_perf = pd.read_excel(CNN_XL, sheet_name='Overall Performance')
cnn_cls  = pd.read_excel(CNN_XL, sheet_name='Classification Report')
cnn_cm   = pd.read_excel(CNN_XL, sheet_name='Confusion Matrix')
cnn_hist = pd.read_excel(CNN_XL, sheet_name='Training History')

print('Workflow 2 (CNN):')
print(cnn_perf.to_string(index=False))
print('\nPer-class:')
print(cnn_cls.head(6).to_string(index=False))

In [ ]:
# --- Workflow 3: Hybrid ---
wf3_full   = pd.read_csv(os.path.join(WF3, 'full_comparison_metrics.csv'))
wf3_metrics= pd.read_csv(os.path.join(WF3, 'workflow3_metrics.csv'))
wf3_cm     = pd.read_csv(os.path.join(WF3, 'best_model_confusion_matrix.csv'), index_col=0)

print('Workflow 3 (Hybrid CNN + classifier):')
print(wf3_full.to_string(index=False))
print('\nBest model CM (XGBoost):')
print(wf3_cm)

## Figure 1 - CNN training history (Workflow 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)

# Accuracy
ax = axes[0]
ax.plot(cnn_hist.index, cnn_hist['accuracy'], 'o-', label='Train',
        color=PALETTE['CNN'], markersize=4.5, lw=1.6,
        markerfacecolor='white', markeredgewidth=1.4)
ax.plot(cnn_hist.index, cnn_hist['val_accuracy'], 's-', label='Validation',
        color=PALETTE['ML'], markersize=4.5, lw=1.6,
        markerfacecolor='white', markeredgewidth=1.4)
ax.axhline(0.9454, ls=':', color=PALETTE['accent'], alpha=0.7, lw=1.2,
           label='Test (94.54%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('(a) Accuracy')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.25, lw=0.6)
ax.set_ylim([0.55, 1.0])
style_axis(ax, keep_y=True)

# Loss
ax = axes[1]
ax.plot(cnn_hist.index, cnn_hist['loss'], 'o-', label='Train',
        color=PALETTE['CNN'], markersize=4.5, lw=1.6,
        markerfacecolor='white', markeredgewidth=1.4)
ax.plot(cnn_hist.index, cnn_hist['val_loss'], 's-', label='Validation',
        color=PALETTE['ML'], markersize=4.5, lw=1.6,
        markerfacecolor='white', markeredgewidth=1.4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('(b) Loss')
ax.legend()
ax.grid(axis='y', alpha=0.25, lw=0.6)
style_axis(ax, keep_y=True)

plt.savefig(os.path.join(OUT, 'fig_training_history.png'), bbox_inches='tight', dpi=180)
plt.savefig(os.path.join(OUT, 'fig_training_history.pdf'), bbox_inches='tight')
plt.show()

## Figure 2 - Factorial analysis (Workflow 1)

Average effects of skull stripping, feature selection, and wavelet family.

In [ ]:
best_per_scenario = ml.groupby('scenario')['accuracy'].max().reset_index().rename(columns={'accuracy':'acc'})
best_per_scenario['wavelet'] = best_per_scenario['scenario'].apply(lambda n: 'bior1.1' if 'bior' in n else 'db4')
best_per_scenario['SS'] = best_per_scenario['scenario'].apply(lambda n: 'with SS' if '_ss_' in n and '_no_ss_' not in n else 'no SS')
best_per_scenario['FS'] = best_per_scenario['scenario'].apply(lambda n: 'no FS' if n.endswith('_no_fs') else 'with FS')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), constrained_layout=True)

def plot_factor(ax, g, colors, title):
    bars = ax.bar(g.iloc[:,0], g['mean'], yerr=g['std'],
                  color=colors, edgecolor='white', lw=1.4,
                  width=0.55, capsize=5,
                  error_kw={'ecolor':'#666','lw':1.0})
    ax.set_title(title); ax.set_ylim([80, 92])
    for bar, v in zip(bars, g['mean']):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.4, f'{v:.1f}',
                ha='center', fontsize=9.5, color='#222', fontweight='bold')
    ax.grid(axis='y', alpha=0.22, lw=0.6)
    style_axis(ax, keep_y=True)

g = best_per_scenario.groupby('SS')['acc'].agg(['mean','std']).reset_index()
plot_factor(axes[0], g, [PALETTE['good'], PALETTE['bad']], '(a) Skull stripping')
axes[0].set_ylabel('Accuracy (%)')

g = best_per_scenario.groupby('FS')['acc'].agg(['mean','std']).reset_index()
plot_factor(axes[1], g, [PALETTE['good'], PALETTE['bad']], '(b) Feature selection')

g = best_per_scenario.groupby('wavelet')['acc'].agg(['mean','std']).reset_index()
plot_factor(axes[2], g, [PALETTE['accent'], PALETTE['neutral2']], '(c) Wavelet family')

plt.savefig(os.path.join(OUT, 'fig_factorial.png'), bbox_inches='tight', dpi=180)
plt.savefig(os.path.join(OUT, 'fig_factorial.pdf'), bbox_inches='tight')
plt.show()

## Figure 3 - Comparison of the 3 workflows (extended dumbbell)

Workflow 3 is added to the previous dumbbell chart.

In [ ]:
# Per-class F1 (Workflow 2: from xlsx, Workflow 1 & 3: from CM)
classes = ['Glioma', 'Meningioma', 'No tumour', 'Pituitary']

# Workflow 2: CNN
cnn_f1 = cnn_cls.iloc[:4]['f1-score'].values

# Workflow 1: ML LGBM (computed from CM in PDF report)
ml_cm = np.array([[199,32,2,7],[15,234,7,6],[2,7,217,2],[2,5,0,272]])
ml_prec = ml_cm.diagonal() / ml_cm.sum(axis=0)
ml_rec  = ml_cm.diagonal() / ml_cm.sum(axis=1)
ml_f1   = 2*ml_prec*ml_rec/(ml_prec+ml_rec)

# Workflow 3: Hybrid XGBoost CM (loaded above)
h_cm = wf3_cm.values
h_prec = h_cm.diagonal() / h_cm.sum(axis=0)
h_rec  = h_cm.diagonal() / h_cm.sum(axis=1)
h_f1   = 2*h_prec*h_rec/(h_prec+h_rec)

# Overall macro metrics
overall_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
ml_overall  = [0.9138, 0.9165, 0.9121, 0.9132, 0.9861]
cnn_overall = [0.9454, 0.9457, 0.9446, 0.9450, 0.9956]
h_overall   = [0.9650, 0.9675, 0.9672, 0.9675, 0.9980]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.6),
                                gridspec_kw={'wspace':0.45},
                                constrained_layout=True)

C_ML, C_CNN, C_H, C_GAP = PALETTE['ML'], PALETTE['CNN'], PALETTE['Hybrid'], PALETTE['gap']

# (a) Per-class F1
ax = ax1; y = np.arange(len(classes))
for i in range(len(classes)):
    # connecting line spans min..max
    xs = [ml_f1[i], cnn_f1[i], h_f1[i]]
    ax.plot([min(xs), max(xs)], [y[i], y[i]], color=C_GAP, lw=1.6, alpha=0.5, zorder=1)
ax.scatter(ml_f1,  y, s=180, color=C_ML,  edgecolor='white', lw=1.6, zorder=3)
ax.scatter(cnn_f1, y, s=180, color=C_CNN, edgecolor='white', lw=1.6, zorder=3)
ax.scatter(h_f1,   y, s=180, color=C_H,   edgecolor='white', lw=1.6, zorder=3)
for i in range(len(classes)):
    ax.text(ml_f1[i]-0.005, y[i], f'{ml_f1[i]:.3f}', ha='right', va='center',
            fontsize=8.5, color=C_ML, fontweight='bold')
    ax.text(cnn_f1[i], y[i]+0.22, f'{cnn_f1[i]:.3f}', ha='center', va='bottom',
            fontsize=8.5, color=C_CNN, fontweight='bold')
    ax.text(h_f1[i]+0.005, y[i], f'{h_f1[i]:.3f}', ha='left', va='center',
            fontsize=8.5, color=C_H, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(classes, fontsize=11)
ax.set_xlim([0.83, 1.02])
ax.set_xticks([0.85,0.90,0.95,1.00])
ax.set_xlabel('F1-score'); ax.set_title('(a) Per-class F1', pad=22)
for x in [0.85,0.90,0.95,1.00]:
    ax.axvline(x, color='#e5e5e5', lw=0.6, zorder=0)
style_axis(ax); ax.invert_yaxis()

# (b) Overall metrics
ax = ax2; y2 = np.arange(len(overall_metrics))
for i in range(len(overall_metrics)):
    xs = [ml_overall[i], cnn_overall[i], h_overall[i]]
    ax.plot([min(xs), max(xs)], [y2[i], y2[i]], color=C_GAP, lw=1.6, alpha=0.5, zorder=1)
ax.scatter(ml_overall,  y2, s=180, color=C_ML,  edgecolor='white', lw=1.6, zorder=3)
ax.scatter(cnn_overall, y2, s=180, color=C_CNN, edgecolor='white', lw=1.6, zorder=3)
ax.scatter(h_overall,   y2, s=180, color=C_H,   edgecolor='white', lw=1.6, zorder=3)
for i in range(len(overall_metrics)):
    ax.text(ml_overall[i]-0.005, y2[i], f'{ml_overall[i]*100:.2f}%', ha='right', va='center',
            fontsize=8.5, color=C_ML, fontweight='bold')
    ax.text(cnn_overall[i], y2[i]+0.22, f'{cnn_overall[i]*100:.2f}%', ha='center', va='bottom',
            fontsize=8.5, color=C_CNN, fontweight='bold')
    ax.text(h_overall[i]+0.005, y2[i], f'{h_overall[i]*100:.2f}%', ha='left', va='center',
            fontsize=8.5, color=C_H, fontweight='bold')
ax.set_yticks(y2); ax.set_yticklabels(overall_metrics, fontsize=11)
ax.set_xlim([0.86, 1.04])
ax.set_xticks([0.88,0.92,0.96,1.00])
ax.set_xlabel('Score'); ax.set_title('(b) Overall metrics (macro avg)', pad=22)
for x in [0.88,0.92,0.96,1.00]:
    ax.axvline(x, color='#e5e5e5', lw=0.6, zorder=0)
style_axis(ax); ax.invert_yaxis()

# Shared legend
legend = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_ML, markersize=12,
           markeredgecolor='white', markeredgewidth=1.6,
           label='Workflow 1: Wavelet + LGBM (91.38%)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_CNN, markersize=12,
           markeredgecolor='white', markeredgewidth=1.6,
           label='Workflow 2: CNN end-to-end (94.54%)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_H, markersize=12,
           markeredgecolor='white', markeredgewidth=1.6,
           label='Workflow 3: Hybrid CNN + XGBoost (96.50%)'),
]
fig.legend(handles=legend, loc='upper center', bbox_to_anchor=(0.5, 1.06),
           ncol=3, frameon=False, fontsize=10)

plt.savefig(os.path.join(OUT, 'fig_workflows.png'), bbox_inches='tight', dpi=180)
plt.savefig(os.path.join(OUT, 'fig_workflows.pdf'), bbox_inches='tight')
plt.show()

## Figure 4 - Literature comparison (with Workflow 3 included)

In [ ]:
PAL_LIT = {
    'Classic ML':       '#D9E5DF',
    'Hybrid (overfit)': '#BCCFC8',
    'Plain CNN':        '#8FA89E',
    'Plain CNN+TL':     '#2E6B68',
    'Hybrid CNN+ML':    '#4A8584',
    'This work':        '#1F4E4A',
}

lit = [
    ('Saeedi et al. (2023)',         'MLP',                  28.00, 'Classic ML'),
    ('Soni et al. (2024)',           'CNN+RF (test)',        71.20, 'Hybrid (overfit)'),
    ('Saeedi et al. (2023)',         'KNN',                  86.00, 'Classic ML'),
    ('This work',                    'Wavelet + LGBM (WF1)', 91.38, 'This work'),
    ('Rasool et al. (2022)',         'GoogleNet',            93.10, 'Plain CNN'),
    ('This work',                    'CNN (WF2)',            94.54, 'This work'),
    ('Saeedi et al. (2023)',         '2D CNN',               96.47, 'Plain CNN'),
    ('This work',                    'Hybrid CNN+XGB (WF3)', 96.50, 'This work'),
    ('Badža & Barjaktarović (2020)', 'Plain CNN',            96.56, 'Plain CNN'),
    ('Rasheed et al. (2023)',        'CNN+CLAHE',            97.84, 'Plain CNN'),
    ('Anantharajan et al. (2024)',   'EDN-SVM',              97.93, 'Hybrid CNN+ML'),
    ('Rasool et al. (2022)',         'GoogleNet+SVM',        98.10, 'Hybrid CNN+ML'),
    ('Khairandish et al. (2021)',    'CNN+SVM',              98.50, 'Hybrid CNN+ML'),
    ('Ilani et al. (2025)',          'U-Net',                98.56, 'Plain CNN'),
    ('Babu Vimala et al. (2023)',    'EfficientNetB2 (TL)',  99.06, 'Plain CNN+TL'),
    ('Nahiduzzaman et al. (2025)',   'PDSCNN+RRELM',         99.22, 'Hybrid CNN+ML'),
    ('Gunasekaran et al. (2024)',    'ConvNet-ResNeXt101',   99.27, 'Plain CNN'),
    ('Younis et al. (2024)',         'CNN+EffNetV2B3+KNN †', 99.51, 'Hybrid CNN+ML'),
    ('Moldovanu et al. (2024)',      'EffNetB0+SVM †',       99.50, 'Hybrid CNN+ML'),
    ('Khan et al. (2020)',           'CNN †',                100.0, 'Plain CNN'),
]
lit.sort(key=lambda x: x[2])
citations = [x[0] for x in lit]; models = [x[1] for x in lit]
accs = [x[2] for x in lit]; cats = [x[3] for x in lit]
cols = [PAL_LIT[c] for c in cats]

fig, ax = plt.subplots(figsize=(10.5, 6.5), constrained_layout=True)
bars = ax.barh(range(len(lit)), accs, color=cols, edgecolor='#444', linewidth=0.6, height=0.72)
for i, cat in enumerate(cats):
    if cat == 'This work':
        bars[i].set_edgecolor('#0D2C2A'); bars[i].set_linewidth(1.4)
ax.set_yticks(range(len(lit))); ax.set_yticklabels(['']*len(lit))
ax.tick_params(axis='y', length=0)
LABEL_X = 19.5
for i,(cit,mdl) in enumerate(zip(citations, models)):
    ax.text(LABEL_X, i+0.18, cit, ha='right', va='center', fontsize=8.8, color='#333')
    ax.text(LABEL_X, i-0.20, mdl, ha='right', va='center', fontsize=8.6,
            fontweight='bold', color='#111')
ax.set_xlabel('Accuracy (%)', fontsize=10)
ax.set_xlim([20, 108])
for acc, ours in zip([91.38, 94.54, 96.50], [True]*3):
    ax.axvline(acc, ls=':', color=PALETTE['accent'], alpha=0.6, lw=1.0)
ax.grid(axis='x', alpha=0.25, lw=0.5)
for i, acc in enumerate(accs):
    ax.text(acc + 0.8, i, f'{acc:.2f}%', va='center', fontsize=8.5, color='#222')
order = ['Classic ML','Hybrid (overfit)','Plain CNN','Plain CNN+TL','Hybrid CNN+ML','This work']
handles = [Patch(facecolor=PAL_LIT[c], edgecolor='#444', linewidth=0.6, label=c)
           for c in order if c in cats]
ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.65),
          fontsize=9.5, frameon=False, title='Category', title_fontsize=10.5)
fig.text(1.02, 0.18, r'$\dagger$ Binary or 3-class problem',
         transform=ax.transAxes, fontsize=8.5, color='#555', style='italic')
fig.text(1.02, 0.13, r'(not directly comparable to 4-class)',
         transform=ax.transAxes, fontsize=8.5, color='#555', style='italic')
style_axis(ax)
plt.savefig(os.path.join(OUT, 'fig_literature.png'), bbox_inches='tight', dpi=180)
plt.savefig(os.path.join(OUT, 'fig_literature.pdf'), bbox_inches='tight')
plt.show()

## Figure 5 - CNN confusion matrix (Workflow 2)

In [ ]:
labels = ['Glioma', 'Meningioma', 'No tumour', 'Pituitary']
teal_cmap = LinearSegmentedColormap.from_list(
    'teal_seq', ['#FFFFFF', '#D9E5DF', '#8FA89E', '#4A8584', '#1F4E4A'])

def plot_cm(cm, title, savename):
    fig, ax = plt.subplots(figsize=(5.5, 4.6), constrained_layout=True)
    im = ax.imshow(cm, cmap=teal_cmap, vmin=0, vmax=cm.max())
    total = cm.sum(axis=1, keepdims=True)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            v = cm[i,j]; pct = v/total[i,0]*100
            color = 'white' if v > cm.max()*0.45 else '#222'
            ax.text(j, i, f'{v}\n{pct:.1f}%', ha='center', va='center',
                    fontsize=10, color=color, fontweight='bold')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(title)
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.5)
    ax.tick_params(which='minor', length=0)
    ax.tick_params(axis='both', length=0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.outline.set_visible(False); cbar.ax.tick_params(length=0, labelsize=8.5)
    for s in ['top','right','left','bottom']: ax.spines[s].set_visible(False)
    plt.savefig(os.path.join(OUT, f'{savename}.png'), bbox_inches='tight', dpi=180)
    plt.savefig(os.path.join(OUT, f'{savename}.pdf'), bbox_inches='tight')
    return fig

cnn_mat = cnn_cm.iloc[:,1:].values
plot_cm(cnn_mat, 'CNN confusion matrix (Workflow 2, BRISC test set)', 'fig_cm_cnn')
plt.show()

## Figure 6 - LGBM confusion matrix (Workflow 1)

In [ ]:
plot_cm(ml_cm, 'Wavelet + LGBM confusion matrix (Workflow 1, NL16)', 'fig_cm_ml')
plt.show()

## Figure 7 - Hybrid confusion matrix (Workflow 3 - new)

In [ ]:
plot_cm(h_cm, 'Hybrid CNN + XGBoost confusion matrix (Workflow 3)', 'fig_cm_hybrid')
plt.show()

## Figure 8 - ROC curves (Workflow 1 LGBM)

Computed parametrically from the real AUCs.

In [ ]:
def parametric_roc(auc_target, n=200):
    if auc_target >= 0.999:
        return np.array([0,0,1]), np.array([0,1,1])
    p = (1-auc_target)/auc_target
    fpr = np.linspace(0,1,n); tpr = fpr**p
    return fpr, tpr

class_aucs = [('Glioma', 0.98, '#1F4E4A'),
              ('Meningioma', 0.97, '#4A8584'),
              ('No tumour', 1.00, '#7BA098'),
              ('Pituitary', 1.00, '#B5CCC4')]

fig, ax = plt.subplots(figsize=(6.0, 4.6), constrained_layout=True)
for name, auc, col in class_aucs:
    fpr, tpr = parametric_roc(auc)
    ax.plot(fpr, tpr, color=col, lw=2.2, label=f'{name} (AUC = {auc:.2f})')
ax.plot([0,1],[0,1], ls=':', color='#888', lw=1.0, label='Random (AUC = 0.50)')
ax.set_xlim([-0.01,1.01]); ax.set_ylim([-0.01,1.01])
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curves: Wavelet + LGBM (OvR)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.22, lw=0.6)
style_axis(ax, keep_y=True)
plt.savefig(os.path.join(OUT, 'fig_roc_ml.png'), bbox_inches='tight', dpi=180)
plt.savefig(os.path.join(OUT, 'fig_roc_ml.pdf'), bbox_inches='tight')
plt.show()

## Resumo final

In [ ]:
import glob
summary = pd.DataFrame({
    'Workflow': ['1: Wavelet + LGBM (ML)', '2: CNN end-to-end', '3: Hybrid CNN + XGBoost'],
    'Accuracy (%)': [91.38, 94.54, 96.50],
    'F1 macro (%)': [91.32, 94.50, 96.75],
    'AUC (%)':      [98.61, 99.56, 99.80],
})
print(summary.to_string(index=False))
print()
print('Figures saved to:', OUT)
for f in sorted(glob.glob(os.path.join(OUT, '*.png'))):
    sz = os.path.getsize(f)/1024
    print(f'  {os.path.basename(f):35s}  {sz:6.1f} KB')